# Act II — The Valley: schema-first, resolution, drift

> *"This is the heart. Slow down. The most-skipped, most-important act."*

The cool graph from Act I was the easy part. Now the valley: schemaless extraction looks
brilliant at small N and quietly falls apart at scale — **the cake is a lie.** Three techniques
get us across:

1. **schema-first** — put the schema + standardised units *in the prompt* (recipe model v1 → v2).
2. *(evolving schemas — pack-only this run)*
3. **entity resolution** — the power-up: get the entities right and the relationships fall out.

We open on the **drift viz** — the slide *into* the valley — then climb out.

Everything here runs **offline**: extraction replays from a committed cache in `data/cache/`,
so there's no API key and no network call on stage.

## The slide in — the drift viz

**[PRINCIPLE]** schemaless looks great at 10 recipes and blurs at 10,000.

**[HOW · recipe + DRIFT VIZ]** As we ingest more recipes with *no resolution*, the count of distinct
ingredient **surface forms** (`"2 cloves garlic, minced"`, `"finely chopped fresh garlic"`, `"Minced
Garlic"` — all different nodes) climbs faster than the count of **canonical** ingredients. The widening
gap is the **duplicate smear** that resolution fixes.

**Be honest about the shape:** this is *not* a plateau — canonical ingredients keep growing too, just
slower. The truthful claim is *"duplicates inflate faster than real ingredients, and the gap widens
with N"*. (We spiked a graph-*fragmentation* story first; it muddies — shared pantry staples bind
recipes into one component — so we pivoted to this duplicate-collapse view, which reads.)

> Note: the figure is computed **live** from the real corpus. We cap N at **1,000** here so the
> notebook runs in well under a minute — the full 10k fuzzy-bucketing is slow and the trend is already
> legible by 1k (the smear ratio climbs monotonically: ×1.11 at N=10 → ×1.36 at N=1k → ×1.50 at 10k).

In [ ]:
from IPython.display import Image

from graphtools.drift import drift_figure

# Live-computed Option C figure. scales capped at 1k for notebook speed (full 10k is the
# committed measurement). Defaults to the real corpus on disk (local 10k if present, else
# the committed 200-row sample) — all offline.
drift_png = "act2_drift.png"
drift_figure(scales=(10, 100, 1000), path=drift_png)
Image(filename=drift_png)

## Technique 1 — schema-first *(when an ontology exists)*

**[PRINCIPLE]** do the schema first for better resolution.

**[HOW · recipe, live]** Same extractor, but now the **schema + standardised-units instruction** lives
in the agent prompt (`extract_recipe_v2`). Output conforms: mass → grams, volume → millilitres, and
clean lowercase singular **canonical names** — instead of the raw `cups`/`tbsp`/`lb` and `Title Case`
surface forms v1 emitted verbatim. This is recipe model **v1 → v2**.

Both extractors **replay from cache** (separate `recipe-v1` / `recipe-v2` cache tags) — no LLM call.

In [ ]:
from graphtools.data import load_hero_texts
from graphtools.extract import extract_recipe, extract_recipe_v2

hero_texts = load_hero_texts()  # ~14 real recipes (TheMealDB), our deterministic corpus

# Hero recipe: Beef Lo Mein — rich in cups/tbsp/lb units and Title-Case plurals, so the
# v1 -> v2 difference is vivid. Both calls are OFFLINE replays from committed caches.
hero_title, hero_text = next((t, x) for t, x in hero_texts if t == "Beef Lo Mein")
v1 = extract_recipe(hero_text)
v2 = extract_recipe_v2(hero_text)

print(f"=== {hero_title}:  v1 (naive)  ->  v2 (schema-first) ===\n")
print(f"  {'ingredient (v1)':22} {'qty/unit (v1)':14} | {'ingredient (v2)':22} {'qty/unit (v2)':14}")
print(f"  {'-'*22} {'-'*14} | {'-'*22} {'-'*14}")
for a, b in zip(v1.ingredients, v2.ingredients):
    u1 = f"{a.quantity if a.quantity is not None else '-'} {a.unit or '-'}"
    u2 = f"{b.quantity if b.quantity is not None else '-'} {b.unit or '-'}"
    flag = "  <- fixed" if (a.unit != b.unit or a.name.lower() != b.name.lower()) else ""
    print(f"  {a.name:22} {u1:14} | {b.name:22} {u2:14}{flag}")

In [ ]:
# The headline: raw cooking units became canonical SI; Title-Case plurals became clean names.
units_v1 = sorted({i.unit for i in v1.ingredients if i.unit})
units_v2 = sorted({i.unit for i in v2.ingredients if i.unit})
print(f"units  v1: {units_v1}")
print(f"units  v2: {units_v2}\n")

print("a few name + unit fixes the schema-in-the-prompt bought us:")
for a, b in zip(v1.ingredients, v2.ingredients):
    if a.unit != b.unit or a.name.lower() != b.name.lower():
        print(f"  {a.name} ({a.quantity} {a.unit})  ->  {b.name} ({b.quantity} {b.unit})")

## Technique 3 — entity resolution *(the power-up)*

**[PRINCIPLE]** if you take one thing away: **resolve your entities.** *I say potato, you say… deb.*
Get the entities right and the relationships often fall out.

**[HOW · recipe, live]** `normalise_ingredient` maps surface forms to canonical names via a curated
synonym table + a rapidfuzz fuzzy fallback. Across the **whole hero set** it collapses Title-Case
plurals, prep adjectives and synonyms onto shared canonical nodes — duplicate ingredient nodes merge,
so the `CONTAINS` edges from different recipes finally point at the *same* node.

And when a lookup table runs out of road, `hybrid_lookup` (vector + lexical) scales resolution past it —
matching `"AP flour"` to canonical `flour` with no shared synonym entry.

In [ ]:
from collections import defaultdict

from graphtools.resolve import normalise_ingredient

# Take every ingredient surface form the v1 extractor produced across the hero set.
recipes_v1 = [extract_recipe(t) for _, t in hero_texts]
surface_forms = [ing.name for r in recipes_v1 for ing in r.ingredients]

distinct_before = set(surface_forms)
distinct_after = {normalise_ingredient(s) for s in surface_forms}
print(f"{len(surface_forms)} ingredient mentions across {len(recipes_v1)} recipes")
print(f"distinct surface forms (before): {len(distinct_before)}")
print(f"distinct canonical    (after) : {len(distinct_after)}")
print(f"-> {len(distinct_before) - len(distinct_after)} duplicate nodes collapsed\n")

# Show the merges concretely: which surface forms folded onto each canonical node.
groups = defaultdict(set)
for s in distinct_before:
    groups[normalise_ingredient(s)].add(s)
merged = {k: v for k, v in groups.items() if len(v) > 1}
print(f"{len(merged)} canonical ingredients absorbed >1 surface form:")
for canon, variants in sorted(merged.items()):
    print(f"  {canon:14} <- {sorted(variants)}")

**[SCORECARD · see it, don't just count it]** That collapse is real but it's a *number* — and a
ten-node drop is invisible in a 250-node hairball. So here's the same move on a legible scale: five
recipes that each spell the shared pantry items differently (`Minced Garlic` / `garlic` / `Garlic
Clove`; `Cumin` / `Cumin seeds`; `Oil` / `vegetable oil`), projected to just the ingredients that drift.

The recipes sit on the **same ring before and after**, so the duplicate spellings on the rim
**collapse onto one shared hub** in the middle — and the recipes, which barely touched, become a
woven web. *That's the relationships falling out of resolved entities.* The cell below computes that
collapse; the animated render is in the talk deck.

In [ ]:
from graphtools.viz_export import RUNG3_RECIPES, drift_focus_graph

# Same resolution, made VISIBLE. The 250-node graph hides a 112->102 collapse, so we
# take a hand-picked handful of recipes that name the SAME pantry items differently and
# project to just the ingredients that drift. Built live, offline (replay from cache).
before = drift_focus_graph(RUNG3_RECIPES, normalise=False)
after = drift_focus_graph(RUNG3_RECIPES, normalise=True)


def ingredient_nodes(g):
    return [n for n, d in g.nodes(data=True) if d["kind"] == "ingredient"]


print(f"{len(RUNG3_RECIPES)} recipes, drift-focus projection:")
print(f"  ingredient nodes  before -> after :  {len(ingredient_nodes(before))} -> {len(ingredient_nodes(after))}")

# The payoff: recipes now SHARE ingredient nodes. An in-degree >= 2 ingredient is a hub
# multiple recipes point at — the relationships that "fall out" once entities are resolved.
hubs = sorted(after.nodes[n]["label"] for n in ingredient_nodes(after) if after.in_degree(n) >= 2)
print(f"  shared ingredient hubs (>=2 recipes): {hubs}")
print("\nbefore: every recipe brought its own spelling, so almost nothing was shared.")
print("after:  recipes converge on those shared canonical hubs — the woven web.")
print("(The animated before/after render of these two graphs is in the talk deck.)")

In [ ]:
from graphtools.resolve import hybrid_lookup

# When the synonym table has no entry, hybrid (vector + lexical) lookup still resolves it.
# 'AP flour' isn't a literal synonym here, yet it ranks canonical 'flour' top.
candidates = sorted(distinct_after)
ranked = hybrid_lookup("AP flour", candidates, top=3)
print(f"hybrid_lookup('AP flour', <{len(candidates)} canonical ingredients>) -> {ranked}")
print(f"\ntop match: {ranked[0]!r}  (no shared synonym-table entry — earned by vector + lexical score)")

---

**[MONEY · judgements]** Same loop on a real corpus: resolved parties, cases and statutes —
the graph finally holds together instead of smearing into duplicates. *(Pre-built, toured in Act III.)*

> *"That's the climb. You're across the valley."* → **Act III is the payoff: algorithms.**